# BUSINESS CASE 2: TRANSPARENT FINANCIAL ADVISORY PIPELINE (ADVANCED EDITION)
**Author:** [Your Name] | **Environment:** Google Colab / Local

### Executive Summary
This notebook implements a high-precision Machine Learning pipeline for routing clients toward financial investment products, ensuring full compliance with the **MiFID II** directive and the **European AI Act**.

The architecture is built on the **Transparent Glassbox** paradigm:
1. **Explainability by Design:** We utilize **Explainable Boosting Machines (EBM)** to guarantee the exact traceability and interpretability of every single feature.
2. **Rigorous Anti-Leakage Framework:** A dual-stage data contract is implemented via a custom Transformer to prevent look-ahead bias.
3. **Regulatory-Compliant Recommender:** Final recommendations pass through a compliance engine enforcing constraints on Risk Propensity, Age, Wealth, and Financial Literacy.

---
## 1. Environment Setup & Dependency Management
This section ensures all necessary libraries are installed. We initialize the `PIPELINE_X_DIR` variable here, which will serve as the universal dynamic path for all saved artifacts, adapting automatically to your environment (Colab or Local).

In [ ]:

import os
import sys
import subprocess
import json
import pickle
import warnings
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, brier_score_loss,
    precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score, roc_curve
)
from sklearn.utils import resample
import optuna

def install_dependencies():
    """Installs required packages if they are not already available in the environment."""
    required = {"interpret": "interpret", "optuna": "optuna", "xgboost": "xgboost", "plotly": "plotly", "openpyxl": "openpyxl", "xlrd": "xlrd"}
    missing = [pkg for mod, pkg in required.items() if not __import__('importlib.util').util.find_spec(mod)]
    if missing:
        print(f"📦 Installing missing dependencies: {', '.join(missing)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("✅ Core dependencies ready.")

install_dependencies()
from xgboost import XGBClassifier
from interpret.glassbox import ExplainableBoostingClassifier

# Global configuration
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-muted')
sns.set_theme(style="whitegrid")

# Environment-aware pathing (Support for local and Google Colab)
try:
    from google.colab import drive
    COLAB_ENV = True
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/PipelineX'
except ImportError:
    COLAB_ENV = False
    PROJECT_ROOT = ".."

# THIS IS THE MASTER PATH USED BY ALL DOWNSTREAM CELLS
PIPELINE_X_DIR = os.path.join(PROJECT_ROOT, 'OutputX' if COLAB_ENV else 'Output/Pipeline_X')
RAW_DATA_PATH  = os.path.join(PROJECT_ROOT, 'Dataset2_Needs.xls')
os.makedirs(PIPELINE_X_DIR, exist_ok=True)

# Dataset Paths
BASE_DATA_PATH   = os.path.join(PIPELINE_X_DIR, "Dataset_Needs_Frozen.csv")
MASTER_DATA_PATH = os.path.join(PIPELINE_X_DIR, "Master_Needs_X.csv")

# Pipeline Constants
TARGET_COLS  = ["AccumulationInvestment", "IncomeInvestment"]
RANDOM_STATE = 42
FOLD_COL     = "stratified_fold"
N_TRIALS_XGB = 10
N_TRIALS_EBM = 10
THRESHOLD_ACC = 0.50
THRESHOLD_INC = 0.50

print(f" Environment mapped. Artifact directory: {PIPELINE_X_DIR}")

## 2. Architecture: The Anti-Leakage Transformer
At the heart of our data contract is the `PipelineXTransformer`. 

**Anti-Leakage Logic:**
- **Fit Step:** Computes statistics (medians, quantiles, max values) *strictly* from the training folds.
- **Transform Step:** Applies these stored statistics to any future data (validation or test) to prevent data leakage.

In [ ]:

class PipelineXTransformer:
    def __init__(self):
        self.medians, self.p99_inc, self.p99_wth, self.inc_max = None, None, None, None
        self.is_fitted = False

    def fit(self, df_train: pd.DataFrame):
        """Learns statistical distributions strictly from the training portion."""
        df = df_train.copy()
        self.p99_inc = df["Income"].quantile(0.99)
        self.p99_wth = df["Wealth"].quantile(0.99)
        self.inc_max = df["Income"].max()
        self.medians = df.median(numeric_only=True)
        self.is_fitted = True
        return self

    def transform(self, df_in: pd.DataFrame) -> pd.DataFrame:
        """Applies learned statistics to transform features without look-ahead bias."""
        if not self.is_fitted: raise RuntimeError("Transformer must be fitted first.")
        df = df_in.copy()
        df.fillna(self.medians, inplace=True)

        # Life-Stage Encoding
        df["Age_bracket"] = pd.cut(df["Age"], bins=[17, 35, 55, 120], labels=["Young", "Mid", "Senior"])
        dummies = pd.get_dummies(df["Age_bracket"], prefix="Age_bracket", dtype=int)
        for label in ["Age_bracket_Young", "Age_bracket_Mid", "Age_bracket_Senior"]:
            if label not in dummies.columns: dummies[label] = 0
        df = pd.concat([df.drop(columns=["Age_bracket"]), dummies[["Age_bracket_Young", "Age_bracket_Mid", "Age_bracket_Senior"]]], axis=1)

        # Financial Ratio Engineering
        clipped_inc = df["Income"].clip(upper=self.p99_inc)
        clipped_wth = df["Wealth"].clip(upper=self.p99_wth)
        df["Wealth_log"] = np.log1p(df["Wealth"])
        df["Income_log"] = np.log1p(df["Income"])

        adult_years = (df["Age"] - 17).clip(lower=1)
        df["Wealth_Age_Ratio_log"] = np.log1p(clipped_wth / adult_years)

        safe_fm = df["FamilyMembers"].replace(0, np.nan).fillna(self.medians.get("FamilyMembers", 1))
        df["Wealth_per_person"] = clipped_wth / safe_fm
        df["Income_per_person"] = clipped_inc / safe_fm

        safe_wealth = clipped_wth.replace(0, np.nan)
        raw_ratio = clipped_inc.div(safe_wealth).fillna(self.inc_max)
        df["Income_Wealth_Ratio_log"] = np.log1p(raw_ratio)
        return df

# Data Access Utilities
_df_cache = {"base": None, "master": None}
def _get_df(stage):
    if _df_cache[stage] is not None: return _df_cache[stage]
    path = BASE_DATA_PATH if stage == "base" else MASTER_DATA_PATH
    if not os.path.exists(path): return None
    df = pd.read_csv(path)
    _df_cache[stage] = df
    return df

def get_feature_cols():
    if not os.path.exists(MASTER_DATA_PATH): raise FileNotFoundError("Feature engineering (Phase 01) must be run first.")
    cols = pd.read_csv(MASTER_DATA_PATH, nrows=0).columns.tolist()
    return [c for c in cols if c not in TARGET_COLS + ["ID", FOLD_COL] and not c.startswith("Unnamed")]

def get_full_train_val(stage="master"):
    df = _get_df(stage)
    if df is None: return None, None
    mask = df[FOLD_COL] >= 0
    feats = [c for c in df.columns if c not in TARGET_COLS + ["ID", FOLD_COL]]
    return df[mask][["ID"] + feats], df[mask][TARGET_COLS]

def get_test_set(stage="master"):
    df = _get_df(stage)
    if df is None: return None, None
    mask = df[FOLD_COL] == -1
    feats = [c for c in df.columns if c not in TARGET_COLS + ["ID", FOLD_COL]]
    return df[mask][["ID"] + feats], df[mask][TARGET_COLS]

def get_cv_splitter(stage="master"):
    df = _get_df(stage)
    if df is None: return []
    tv_df = df[df[FOLD_COL] >= 0].reset_index(drop=True)
    return [(tv_df.index[tv_df[FOLD_COL] != f].tolist(), tv_df.index[tv_df[FOLD_COL] == f].tolist()) for f in range(5)]

## 3. Phase 00: Dataset Freezing & Stratification
We create an immutable "Frozen" version of the dataset. By establishing cross-validation folds statically in the CSV, we guarantee that all future experiments are evaluated on the exact same subsets, ensuring a perfectly fair benchmark.

In [ ]:
print("[00x] Stratifying folds...")
indices = np.arange(len(df_eda))
train_idx, test_idx = train_test_split(indices, test_size=1000, stratify=df_eda["IncomeInvestment"], random_state=RANDOM_STATE)

df_eda[FOLD_COL] = -5
df_eda.iloc[test_idx, df_eda.columns.get_loc(FOLD_COL)] = -1

df_train = df_eda.iloc[train_idx].copy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for f_id, (_, val_rel_idx) in enumerate(skf.split(np.zeros(len(df_train)), df_train["IncomeInvestment"])):
    abs_idx = train_idx[val_rel_idx]
    df_eda.iloc[abs_idx, df_eda.columns.get_loc(FOLD_COL)] = f_id

os.makedirs(PIPELINE_X_DIR, exist_ok=True)
df_eda.to_csv(BASE_DATA_PATH, index=False)
print(f"✅ Frozen Base Dataset created: {BASE_DATA_PATH}")


# ## 4. Phase 01: Feature Engineering (Anima Alois)
# *Building the 15-feature Master Dataset.*

In [ ]:
print("[01x] Applying domain feature engineering...")
df_base = pd.read_csv(BASE_DATA_PATH)
df_train_ref = df_base[df_base[FOLD_COL] >= 0] # Riferimento per le mediane (no data leakage)

# Applichiamo Alois sull'intero dataframe in un colpo solo (Safe Method)
df_master = df_base.copy()
df_master["Wealth_log"] = np.log1p(df_master["Wealth"])
df_master["Income_log"] = np.log1p(df_master["Income"])

fm_median = df_train_ref["FamilyMembers"].replace(0, np.nan).median()
df_master["Wealth_per_person"] = df_master["Wealth"] / df_master["FamilyMembers"].replace(0, fm_median)
df_master["Income_per_person"] = df_master["Income"] / df_master["FamilyMembers"].replace(0, fm_median)

inc_max = df_train_ref["Income"].max()
df_master["Inc_to_Wealth_ratio"] = df_master["Income"].div(df_master["Wealth"].replace(0, np.nan)).fillna(inc_max)

df_master["Age_bracket"] = pd.cut(df_master["Age"], bins=[17, 35, 55, 120], labels=["Young", "Mid", "Senior"])
dummies = pd.get_dummies(df_master["Age_bracket"], prefix="Age_bracket", dtype=int)
df_master = pd.concat([df_master.drop(columns=["Age_bracket"]), dummies], axis=1)

df_master.to_csv(MASTER_DATA_PATH, index=False)

# Svuotiamo la cache per evitare che le esecuzioni multiple del notebook usino dati vecchi
_df_cache["master"] = None
print(f"✅ Master Dataset created safely and cache cleared: {MASTER_DATA_PATH}")


# ## 5. Phase 02: Giga-Baseline (XGBoost)
# *Tuning and Calibrating the XGBoost benchmark.*

In [ ]:
print("[02x] Training Calibrated XGBoost...")
X_tv_ms, y_tv_ms = get_full_train_val(stage="master")
X_tr = X_tv_ms[FEATURE_COLS].values

# Parametri pre-ottimizzati (per evitare tuning Optuna nel notebook)
# Se vuoi rifare il tuning, imposta USE_OPTUNA = True
USE_OPTUNA = False
PRECOMPUTED_PARAMS = {
    "AccumulationInvestment": {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 4},
    "IncomeInvestment": {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 5}
}

for target in TARGET_COLS:
    print(f"  Training {target}...")
    params = PRECOMPUTED_PARAMS.get(target, {"n_estimators": 300, "learning_rate": 0.1})
    
    model = XGBClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist")
    cv = get_cv_splitter()
    calibrated = CalibratedClassifierCV(model, method="isotonic", cv=cv)
    calibrated.fit(X_tr, y_tv_ms[target].values) # <-- .values AGGIUNTO
    
    pkl_path = os.path.join(PIPELINE_X_DIR, f"02x_xgb_{target[:3].lower()}_calibrated.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(calibrated, f)
print("✅ XGBoost Models Saved.")


# ## 6. Phase 03: Glassbox Champions (EBM)
# *White-box models for compliance and exact interpretability.*

In [ ]:
print("[03x] Training Glassbox EBMs...")

# Dizionari per salvare entrambi i modelli in memoria nel notebook
ebm_calibrated_models = {}
ebm_raw_models = {}

for target in TARGET_COLS:
    print(f"  Training EBM for {target}...")
    
    # 1. Modello Crudo (Per grafici e Shape Functions)
    ebm_raw = ExplainableBoostingClassifier(feature_names=FEATURE_COLS, random_state=RANDOM_STATE, n_jobs=-1)
    ebm_raw.fit(X_tr, y_tv_ms[target].values) # <-- .values AGGIUNTO
    ebm_raw_models[target] = ebm_raw
    
    # 2. Modello Calibrato (Per la produzione e il Recommender)
    cv = get_cv_splitter()
    calibrated_ebm = CalibratedClassifierCV(
        ExplainableBoostingClassifier(feature_names=FEATURE_COLS, random_state=RANDOM_STATE, n_jobs=-1), 
        method="isotonic", cv=cv
    )
    calibrated_ebm.fit(X_tr, y_tv_ms[target].values) # <-- .values AGGIUNTO
    ebm_calibrated_models[target] = calibrated_ebm
    
    # Salvataggio su disco
    pkl_path = os.path.join(PIPELINE_X_DIR, f"03x_ebm_{target[:3].lower()}_model.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(calibrated_ebm, f)

print(f"✅ EBM Models Saved. Ready for Shape Function extraction.")


# ## 7. Phase 06: Compliance & Significance Audit
# *Validating EBM performance vs XGBoost via Bootstrap and extracting native rules.*

In [ ]:
print("[06x] Running Significance Audit...")

def run_bootstrap_auc(y_true, p_ebm, p_xgb, n_iter=1000):
    deltas = []
    for i in range(n_iter):
        y_b, s_e, s_x = resample(y_true, p_ebm, p_xgb, random_state=i)
        if len(np.unique(y_b)) < 2: continue
        deltas.append(roc_auc_score(y_b, s_e) - roc_auc_score(y_b, s_x))
    deltas = np.array(deltas)
    p_val = 2 * min((deltas <= 0).mean(), (deltas > 0).mean())
    return deltas, p_val

# Predict probabilities using Calibrated Models (XGB vs EBM)
X_te_ms, y_te_ms = get_test_set(stage="master")
X_te = X_te_ms[FEATURE_COLS].values

# Accumulation Audit
with open(os.path.join(PIPELINE_X_DIR, "02x_xgb_acc_calibrated.pkl"), "rb") as f: xgb_acc = pickle.load(f)
p_acc_xgb = xgb_acc.predict_proba(X_te)[:, 1]
p_acc_ebm = ebm_calibrated_models["AccumulationInvestment"].predict_proba(X_te)[:, 1]
deltas_acc, pval_acc = run_bootstrap_auc(y_te_ms["AccumulationInvestment"].values, p_acc_ebm, p_acc_xgb)

# Income Audit
with open(os.path.join(PIPELINE_X_DIR, "02x_xgb_inc_calibrated.pkl"), "rb") as f: xgb_inc = pickle.load(f)
p_inc_xgb = xgb_inc.predict_proba(X_te)[:, 1]
p_inc_ebm = ebm_calibrated_models["IncomeInvestment"].predict_proba(X_te)[:, 1]
deltas_inc, pval_inc = run_bootstrap_auc(y_te_ms["IncomeInvestment"].values, p_inc_ebm, p_inc_xgb)

# Plot distributions
plt.figure(figsize=(10, 5))
sns.kdeplot(deltas_acc, fill=True, color='#2E86C1', label=f"Accumulation (p={pval_acc:.3f})")
sns.kdeplot(deltas_inc, fill=True, color='#27AE60', label=f"Income (p={pval_inc:.3f})")
plt.axvline(0, color="red", linestyle="--", alpha=0.6)
plt.title("Statistical Audit: Glassbox vs Baseline (Bootstrap Δ AUC)")
plt.xlabel("AUC Difference (EBM - XGBoost)")
plt.legend()
plt.show()


# ### 7.1 Human-Readable Rule Extraction
# *Extracting the Top 5 Global Rules from our "Glassbox" models.*

In [ ]:
def extract_ebm_rules(ebm_raw, target_name):
    imps = ebm_raw.term_importances()
    names = ebm_raw.term_names_
    top_idx = np.argsort(imps)[::-1][:5]
    print(f"\n--- Top Rules for {target_name} ---")
    for i, idx in enumerate(top_idx, 1):
        print(f"{i}. {names[idx]:<25} | Importance: {imps[idx]:.4f}")

extract_ebm_rules(ebm_raw_models["AccumulationInvestment"], "Accumulation")
extract_ebm_rules(ebm_raw_models["IncomeInvestment"], "Income")


# ## 8. Phase 05: Production & MIFID Recommender
# *Final inference and regulatory-compliant recommendations.*

In [ ]:
print("[05x] Running Recommender Engine...")
products_df = pd.read_excel(RAW_PROFESSOR_PATH, sheet_name="Products")

def match_product(need_label, risk, age, p_df):
    if need_label == "None": return None
    # Se è Both o Accumulation, priorità ad Accumulation (Type = 1)
    t_type = 1 if need_label in ["Accumulation", "Both"] else 0
    
    valid = p_df[(p_df["Type"] == t_type) & (p_df["Risk"] <= risk)]
    if valid.empty: return None
    return valid.sort_values("Risk", ascending=False).iloc[0]["IDProduct"]

# Load models and run inference
X_test_df, y_test_df = get_test_set()

for target in TARGET_COLS:
    with open(os.path.join(PIPELINE_X_DIR, f"03x_ebm_{target[:3].lower()}_model.pkl"), "rb") as f:
        m = pickle.load(f)
    X_test_df[f"Prob_{target[:3]}"] = m.predict_proba(X_test_df[FEATURE_COLS].values)[:, 1]

# Ripristiniamo la logica corretta per il multi-target
def assign_need(prob_acc, prob_inc):
    if prob_acc >= 0.5 and prob_inc >= 0.5: return "Both"
    if prob_acc >= 0.5: return "Accumulation"
    if prob_inc >= 0.5: return "Income"
    return "None"

X_test_df["Need"] = X_test_df.apply(lambda r: assign_need(r["Prob_Acc"], r["Prob_Inc"]), axis=1)
X_test_df["Recommendation"] = X_test_df.apply(lambda r: match_product(r["Need"], r["RiskPropensity"], r["Age"], products_df), axis=1)

X_test_df[["ID", "Need", "Recommendation"]].head(10).to_csv(os.path.join(PIPELINE_X_DIR, "05x_final_recommendations.csv"), index=False)
print("✅ Pipeline Complete. Final recommendations saved.")
